In [ ]:
import os
import sys
import warnings
import time
import json
from natsort import natsorted
import numpy as np
import xarray as xr
import pandas as pd
import math 
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.pyplot import figure
from matplotlib.patches import Patch, Rectangle
import matplotlib.ticker as ticker
from matplotlib.ticker import FuncFormatter
import matplotlib.colors as mcolors
import seaborn as sns
import cmasher as cmr
from pathlib import PureWindowsPath, PurePosixPath
import pickle
from scipy.signal import find_peaks
import scipy.stats as stats

sys.path.append('../utils') 
from utils_plot import (
    set_pub_style, get_asterisks, save_metadata_json, 
    lighten_color, add_stat_annotation_two_sided, add_significance_bar)

#General parameters
dir_stat = r'../../data/09.Anatomy_behavior'
bin_width = 200  # ms
fs = int(1000/bin_width)

colors_anatomy = ['#A6761D', '#845ec2', '#97cebf'] 
colors_beh_i = ['#4091cf', '#e1703c'] # Blue, red
colors_beh_e = ['#4091cf', '#8cba54'] # Blue, green
#plot
dir_output = r'../output_figures'
os.makedirs(dir_output, exist_ok=True)
dir_fig = 'Fig2'
dpath_plot = os.path.join(dir_output, dir_fig)
if not os.path.exists(dpath_plot):
    os.makedirs(dpath_plot)   

## 1 TFC_freezing scores plotting

In [ ]:
dic_plots = {'Normal_I_1mgKg':  ['TFC_C_1mgKg', 'TFC_I_1mgKg'],
            'Normal_E_0.2mgKg': ['TFC_C_0.2mgKg', 'TFC_E_0.2mgKg'],
            'Normal_E_1mgKg': ['TFC_C_1mgKg', 'TFC_E_1mgKg']}
height_mm = 30
idx_plot = 0
for key, plot_groups in dic_plots.items():   
    idx_plot +=1
    if 'I' in key:
        colors = colors_beh_i
    elif 'E' in key:
        colors = colors_beh_e
    if key == 'Normal_E_1mgKg':
        pre_title = 'sup_01'
    else:
        pre_title = '01'
    #1. CondiA curve plotting
    df_stat = pd.read_csv(os.path.join(dir_stat, "02_1.TFC_scores_condi.csv")) 
    event_info_ntfc_condi = {'tone_start': [240, 340, 440, 540], 'tone_len': 20,
                 'shock_start': [280, 380, 480, 580], 'shock_len': 2}
    height_mm = 30
    width_mm = 40  #  inches 
    plot_tfc_condi_curve(df_stat, 'nTFC', plot_groups, width_mm, height_mm, colors, event_info_ntfc_condi,  dpath_plot, f'{pre_title}_{str(idx_plot)}_1_{key}_TFC_condi_curve')
    
    #2. Recall B and statictics
    df_stat = pd.read_csv(os.path.join(dir_stat, "02_2.TFC_scores_recallB.csv")) 
    event_info_ntfc_recall = {'base': 0, 'base_len': 120,
                        'tone_start': [120, 300, 480, 660], 'tone_len': 60,
                         'p_tone_start':[180, 360, 540, 720], 'p_tone_len': 120}
    width_mm= 60 #  inches 
    plot_combined_tfc_recall(df_stat, 'nTFC', plot_groups, width_mm, height_mm, colors, event_info_ntfc_recall, dpath_plot,  f'{pre_title}_{str(idx_plot)}_2_{key}_TFC_recallB_curve_stat')
    
    #3. Recall B+1wk and statictics
    df_stat = pd.read_csv(os.path.join(dir_stat, "02_2.TFC_scores_recallB_1wk.csv"))  
    width_mm= 60 #  inches 
    plot_combined_tfc_recall(df_stat, 'nTFC', plot_groups, width_mm, height_mm, colors, event_info_ntfc_recall, dpath_plot, f'{pre_title}_{str(idx_plot)}_3_{key}_TFC_recallB_1wk_curve_stat')

    #4. CFC & CFC_1wk statisticks
    df_stat = pd.read_csv(os.path.join(dir_stat, "02_3.TFC_scores_recallA&1wk.csv")) 
    width_mm  = 20 #  inches 
    height_mm = 28
    plot_cfc_recall_stat(df_stat, plot_groups, width_mm, height_mm, colors, 'RecallA', dpath_plot, f'{pre_title}_{str(idx_plot)}_4_{key}_TFC_recallA_stat')
    plot_cfc_recall_stat(df_stat, plot_groups, width_mm, height_mm, colors, 'RecallA_1wk', dpath_plot, f'{pre_title}_{str(idx_plot)}_5_{key}_TFC_recallA_1wk_stat')
print('All finished')

In [ ]:
def format_to_minutes(x, pos):
        return f"{int(x / 60)}"
def plot_tfc_condi_curve(df_stat, flag, plot_groups, width_mm, height_mm, colors, event_info, dpath_plot, title):
    """
    Plots a continuous freezing curve over time with SEM shading and event markers.
    
    plot_groups: List of group names to plot, in order (e.g., ['TFC_C_1mgKg', 'TFC_I_1mgKg']).
    event_info: Dictionary containing tone and shock timing.
    """
    set_pub_style()
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')    
    # Sort the original column keys based on their actual numerical value
    if flag=='nTFC':
        time_cols = df_stat.columns[2:-1].values 
    if flag=='wTFC':
        time_cols = df_stat.columns[2:-3].values 
    # Create a strictly numerical array for Matplotlib's X-axis coordinates
    time_vals = np.array([float(t) for t in time_cols])
    
    metadata = {"Figure_Title": title, "Data_Summary": {}}
    # --- 2. PLOT EVENT SHADINGS (Tone & Shock) ---
    tone_starts = event_info.get('tone_start', [])
    tone_len = event_info.get('tone_len', 20)
    for start in tone_starts:
        ax.axvspan(start, start + tone_len, color='#d3d3d3', alpha=0.8, linewidth=0, zorder=1)
        
    shock_starts = event_info.get('shock_start', [])
    shock_len = event_info.get('shock_len', 2)
    for start in shock_starts:
        ax.axvspan(start, start + shock_len, facecolor='red', edgecolor='red', linewidth=0.1, alpha=1.0, zorder=1)
    
    # --- 3. PLOT GROUP CURVES ---
    for i, group in enumerate(plot_groups):       
        # Subset the dataframe using the exact original column names (time_cols)
        group_df = df_stat[df_stat['Group'] == group]
        data_matrix = group_df[time_cols].values.astype(float)
        
        n_mice = data_matrix.shape[0]       
        means = np.nanmean(data_matrix, axis=0)
        sems = stats.sem(data_matrix, axis=0, nan_policy='omit')        
        color = colors[i]
        
        # Plot using the numerical X-axis coordinates (time_vals)
        ax.plot(time_vals, means, color=color, linewidth=0.6, label=f"{group} (N={n_mice})", zorder=3)
        ax.fill_between(time_vals, means - sems, means + sems, color=color, alpha=0.2, linewidth=0, zorder=2)
        
        # Log metadata
        metadata["Data_Summary"][group] = {
            "N_mice": n_mice,
            "Final_Timepoint_Mean": float(means[-1]) if len(means) > 0 else 0,
            "Final_Timepoint_SEM": float(sems[-1]) if len(sems) > 0 else 0
        }
    # --- 4. AESTHETICS & FORMATTING ---
    ax.set_xlabel("Time (min)",  labelpad=0.1)
    ax.set_ylabel("Freezing score (%)", labelpad=0.1)
    
    ax.set_ylim(0, 100)
    ax.yaxis.set_major_locator(ticker.MultipleLocator(20))  
    # 1. Force the X-axis to place a tick every 120 seconds (2 minutes)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(120))    
    # 2. Tell Matplotlib to divide the printed tick label by 60  
    ax.xaxis.set_major_formatter(FuncFormatter(format_to_minutes))
    # Set X limits based on raw seconds
    max_time = max(time_vals) if len(time_vals) > 0 else 690
    ax.set_xlim(0, max_time)   
    #ax.grid(axis='y', color='gray', linestyle='--', linewidth=0.5, alpha=0.3, zorder=0)
    
    # --- 5. SAVING OUTPUTS ---
    base_path = os.path.join(dpath_plot, title.replace(' ', '_'))  
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)
    plt.savefig(f"{base_path}.pdf")   #bbox_inches='tight', pad_inches=0.02
    plt.savefig(f"{base_path}.png", dpi=300) 
    plt.close()    
    save_metadata_json(metadata, dpath_plot, title)

def plot_combined_tfc_recall(df_stat, flag, plot_groups, width_mm, height_mm, colors_beh, event_info, dpath_plot, title):
    """
    Plots a continuous freezing curve (Left Panel) and epoch statistics (Right Panel) side-by-side.
    """
    set_pub_style()  
    # 1. Create a 1x2 grid. width_ratios=[4, 4] ensures both panels take exactly 50% of the space.
    fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(width_mm / 25.4, height_mm / 25.4),
                               gridspec_kw={'width_ratios': [4, 4]}, layout='constrained')   
    if flag == 'wTFC':
        time_cols = df_stat.columns[2:-7].values   
    else:
        time_cols = df_stat.columns[2:-1].values 
    time_vals = np.array([float(t) for t in time_cols])    
    metadata = {"Figure_Title": title, "Curve_Data": {}, "Statistics": {}}
    global_y_max = 100.0 # Will track the highest bracket in the stats plot 
    ax1.set_ylim(0, global_y_max) # For the correct distance between line and stars
    ax2.set_ylim(0, global_y_max)
    # ==========================================
    # PANEL 1: CONTINUOUS CURVE (ax1)
    # ==========================================
    # Event Shadings
    tone_starts = event_info.get('tone_start', [])
    tone_len = event_info.get('tone_len', 60) # Default to 60 for recall
    for start in tone_starts:
        ax1.axvspan(start, start + tone_len, color='#d3d3d3', alpha=0.8, linewidth=0, zorder=1)       
    # Group Curves
    for i, group in enumerate(plot_groups):        
        group_df = df_stat[df_stat['Group'] == group]
        data_matrix = group_df[time_cols].values.astype(float)
        
        n_mice = data_matrix.shape[0]        
        means = np.nanmean(data_matrix, axis=0)
        sems = stats.sem(data_matrix, axis=0, nan_policy='omit')        
        color = colors_beh[i]
        
        ax1.plot(time_vals, means, color=color, linewidth=0.6, label=f"{group} (N={n_mice})", zorder=3)
        ax1.fill_between(time_vals, means - sems, means + sems, color=color, alpha=0.2, linewidth=0, zorder=2)
        
        metadata["Curve_Data"][group] = {
            "N_mice": n_mice,
            "Final_Mean": float(means[-1]) if len(means) > 0 else 0,
            "Final_SEM": float(sems[-1]) if len(sems) > 0 else 0}
    # Ax1 Aesthetics
    ax1.set_xlabel("Time (min)", labelpad=0.1)
    ax1.set_ylabel("Freezing score (%)",  labelpad=0.1)    
    ax1.xaxis.set_major_locator(ticker.MultipleLocator(120))    
    ax1.xaxis.set_major_formatter(FuncFormatter(format_to_minutes))    
    max_time = max(time_vals) if len(time_vals) > 0 else 690
    ax1.set_xlim(0, max_time)    
    # ==========================================
    # PANEL 2: EPOCH STATISTICS (ax2)
    # ==========================================
    base_cols, tone_cols, ptone_cols = [], [], []
    base_start = event_info.get('base', 0)
    base_len = event_info.get('base_len', 120)      
    for t in time_cols:
        val = float(t)        
        if base_start <= val < base_start + base_len:
            base_cols.append(t)
        for start in event_info.get('tone_start', []):
            if start <= val < start + event_info.get('tone_len', 60):
                tone_cols.append(t)
        for start in event_info.get('p_tone_start', []):
            if start <= val < start + event_info.get('p_tone_len', 20):
                ptone_cols.append(t)                    
    epochs = ['Baseline', 'Tone', 'Post']
    epoch_col_map = [base_cols, tone_cols, ptone_cols]
    
    group_spacing = 1.2
    n_groups = len(plot_groups)    
    bar_width = 0.3
    offsets = np.linspace(-0.2, 0.2, n_groups) if n_groups == 2 else np.linspace(-0.3, 0.3, n_groups)
    
    for e_idx, epoch_name in enumerate(epochs):
        metadata["Statistics"][epoch_name] = {}
        base_x = e_idx * group_spacing        
        epoch_group_data = [] 
        epoch_group_x = []      
        
        for g_idx, group in enumerate(plot_groups):
            group_df = df_stat[df_stat['Group'] == group]
            cols = epoch_col_map[e_idx]            
            data_matrix = group_df[cols].values.astype(float)
            
            if data_matrix.size == 0: continue
                
            animal_means = np.nanmean(data_matrix, axis=1) 
            animal_means = animal_means[~np.isnan(animal_means)] 
            epoch_group_data.append(animal_means)
            
            n_mice = len(animal_means)
            if n_mice == 0: continue
            
            x_pos = base_x + offsets[g_idx]
            epoch_group_x.append(x_pos)
            color = colors_beh[g_idx]
            
            mean_val = np.mean(animal_means)
            sem_val = stats.sem(animal_means)            
            metadata["Statistics"][epoch_name][group] = {
                "N_mice": n_mice, "Mean": float(mean_val), "SEM": float(sem_val)}
            
            ax2.bar(x_pos, mean_val, yerr=sem_val, width=bar_width, 
                    facecolor='none', edgecolor=color, linewidth=0.75, capsize=0,
                    error_kw=dict(lw=0.75, ecolor='black'), zorder=2)          
            x_jitter = x_pos + np.random.uniform(-0.06, 0.06, size=n_mice)
            ax2.scatter(x_jitter, animal_means, color=color, edgecolor='none', 
                        s=2, zorder=3, alpha=1.0)
        
        # Apply Statistics
        if len(epoch_group_data) == 2 and len(epoch_group_data[0]) >= 3 and len(epoch_group_data[1]) >= 3:
            local_max = max(np.max(epoch_group_data[0]), np.max(epoch_group_data[1]))
            error_max = max(np.mean(epoch_group_data[0]) + stats.sem(epoch_group_data[0]), 
                            np.mean(epoch_group_data[1]) + stats.sem(epoch_group_data[1]))
            top_y = max(local_max, error_max)

            _, p_value = stats.mannwhitneyu(epoch_group_data[0], epoch_group_data[1], alternative='two-sided')
            metadata["Statistics"][epoch_name]['mannwhitneyu_p-values'] = float(p_value)
            
            bracket_top = add_stat_annotation_two_sided(
                ax2, epoch_group_data[0], epoch_group_data[1], 
                epoch_group_x[0], epoch_group_x[1], y_max=top_y, ttest=0, paired=0)
                
            if bracket_top is not None:
                global_y_max = max(global_y_max, bracket_top)

    # Ax2 Aesthetics
    ax2.set_xticks([i * group_spacing for i in range(len(epochs))])
    ax2.set_xticklabels(epochs, fontsize=7)
    #ax2.set_ylabel("Freezing score (%)", labelpad=0.1)
    # ==========================================
    # FINAL SYNCHRONIZATION & EXPORT
    # ==========================================
    # Sync Y-axis so grids match perfectly across the figure
    ax1.set_ylim(0, global_y_max)
    ax2.set_ylim(0, global_y_max)
    
    ax1.yaxis.set_major_locator(ticker.MultipleLocator(20))
    ax2.yaxis.set_major_locator(ticker.MultipleLocator(20))

    base_path = os.path.join(dpath_plot, title.replace(' ', '_'))      
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)
    plt.savefig(f"{base_path}.pdf")   #bbox_inches='tight', pad_inches=0.02
    plt.savefig(f"{base_path}.png", dpi=300) 
    plt.close()          
    save_metadata_json(metadata, dpath_plot, title)
    
    
def plot_cfc_recall_stat(df_stat, plot_groups, width_mm, height_mm, colors_beh, data_col, dpath_plot, title):
    """
    Calculates the session-wide average freezing score for Contextual Fear Conditioning
    and plots it as a single grouped bar/scatter chart optimized for narrow widths.
    """
    set_pub_style()
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')  
    # --- 2. PLOTTING SETUP ---
    n_groups = len(plot_groups)   
    
    # Dynamically center offsets based on number of groups 
    bar_width = 0.3
    offsets = np.linspace(-0.2, 0.2, n_groups) if n_groups == 2 else np.linspace(-0.3, 0.3, n_groups)
    ax.set_ylim(0, 100)
    
    metadata = {"Figure_Title": title, "Statistics": {"Context": {}}}       
    # --- 3. DATA EXTRACTION & PLOTTING ---
    # Only have one "epoch" (the entire context session), so base_x is simply 0
    base_x = 0    
    epoch_group_data = [] # Store data arrays for stats later
    epoch_group_x = []    # Store exact X coordinates for stats bracket       
    
    for g_idx, group in enumerate(plot_groups):
        group_df = df_stat[df_stat['Group'] == group]
        animal_means = group_df[data_col].values.astype(float)
        animal_means = animal_means[~np.isnan(animal_means)] # Drop true NaNs
        
        epoch_group_data.append(animal_means)
        
        n_mice = len(animal_means)
        if n_mice == 0: continue
        
        x_pos = base_x + offsets[g_idx]
        epoch_group_x.append(x_pos)
        color = colors_beh[g_idx]
        
        mean_val = np.mean(animal_means)
        sem_val = stats.sem(animal_means)                    
        # Record Metadata
        metadata["Statistics"]["Context"][group] = {
            "N_mice": n_mice, "Mean": float(mean_val), "SEM": float(sem_val)}
        
        # A. Draw Bar (Face='none', Edge=color)
        ax.bar(x_pos, mean_val, yerr=sem_val, width=bar_width, 
               facecolor='none', edgecolor=color, linewidth=0.75, capsize=0,
               error_kw=dict(lw=0.75, ecolor='black'), zorder=2)
        
        # B. Scatter Raw Data 
        x_jitter = x_pos + np.random.uniform(-0.06, 0.06, size=n_mice)
        ax.scatter(x_jitter, animal_means, color=color, edgecolor='none', 
                   s=2, zorder=3, alpha=1.0)
    
    # --- 4. APPLY STATISTICS ---
    # If have exactly 2 groups, test them against each other
    if len(epoch_group_data) == 2 and len(epoch_group_data[0]) >= 3 and len(epoch_group_data[1]) >= 3:
        local_max = max(np.max(epoch_group_data[0]), np.max(epoch_group_data[1]))
        error_max = max(np.mean(epoch_group_data[0]) + stats.sem(epoch_group_data[0]), 
                        np.mean(epoch_group_data[1]) + stats.sem(epoch_group_data[1]))
        top_y = max(local_max, error_max)

        _, p_value = stats.mannwhitneyu(epoch_group_data[0], epoch_group_data[1], alternative='two-sided')
        metadata["Statistics"]['mannwhitneyu_p-values'] = float(p_value)
        
        # Call the robust annotation function (using Mann-Whitney by default)
        add_stat_annotation_two_sided(
            ax, epoch_group_data[0], epoch_group_data[1], 
            epoch_group_x[0], epoch_group_x[1], y_max=top_y, ttest=0, paired=0)
            
    # --- 5. AESTHETICS & FORMATTING ---
    ax.set_xticks([base_x])
    ax.set_xticklabels(['Context'], fontsize=7)
    ax.set_ylabel("Freezing score (%)", labelpad=0.1)
    
    # Setup standard 0-100 scale
    ax.yaxis.set_major_locator(ticker.MultipleLocator(20))  
    
    # Force the X-axis limits to hug the bars tightly, preventing massive side margins
    ax.set_xlim(offsets[0] - 0.4, offsets[-1] + 0.4)     
    # --- 6. SAVING OUTPUTS ---
    base_path = os.path.join(dpath_plot, title.replace(' ', '_'))        
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)
    plt.savefig(f"{base_path}.pdf")   #bbox_inches='tight', pad_inches=0.02
    plt.savefig(f"{base_path}.png", dpi=300) 
    plt.close()        
    
    save_metadata_json(metadata, dpath_plot, title)


## 2. wTFC freezing scores plotting

In [ ]:
#For weak trace fear conditioning
dic_plots = {'Weak_I_2mgKg':  ['wTFC_C_2mgKg', 'wTFC_I_2mgKg'],
            'Weak_E_0.2mgKg': ['wTFC_C_0.2mgKg', 'wTFC_E_0.2mgKg']}
height_mm = 30
idx_plot = 0
for key, plot_groups in dic_plots.items():   
    idx_plot +=1
    if 'I' in key:
        colors = colors_beh_i
    elif 'E' in key:
        colors = colors_beh_e
    #1. CondiA curve plotting
    df_stat = pd.read_csv(os.path.join(dir_stat, "02_1.TFC_scores_condi.csv")) 
    event_info_wtfc_condi = {'tone_start': [240, 460], 'tone_len': 20, 'shock_start': [280, 500], 'shock_len': 2}
    height_mm = 30
    width_mm = 35  #  inches 
    plot_tfc_condi_curve(df_stat, 'wTFC', plot_groups, width_mm, height_mm, colors, event_info_wtfc_condi,  dpath_plot, f'02_{str(idx_plot)}_1_{key}_TFC_condi_curve')
    
    #2. Recall B and statictics
    df_stat = pd.read_csv(os.path.join(dir_stat, "02_2.TFC_scores_recallB.csv")) 
    event_info_wtfc_recall = {'base': 0, 'base_len': 180,
                        'tone_start': [180, 330, 480, 630], 'tone_len': 30,
                         'p_tone_start':[210, 360, 510, 660], 'p_tone_len': 120}
    width_mm= 60#  inches 
    plot_combined_tfc_recall(df_stat, 'wTFC', plot_groups, width_mm, height_mm, colors, event_info_wtfc_recall, dpath_plot,  f'02_{str(idx_plot)}_2_{key}_TFC_recallB_curve_stat')

    #4. CFC  statisticks
    df_stat = pd.read_csv(os.path.join(dir_stat, "02_3.TFC_scores_recallA&1wk.csv")) 
    width_mm = 20  #  inches 
    height_mm = 28
    plot_cfc_recall_stat(df_stat, plot_groups, width_mm, height_mm, colors, 'RecallA', dpath_plot, f'02_{str(idx_plot)}_3_{key}_TFC_recallA_stat')
print('All finished')

In [ ]:
def format_to_minutes(x, pos):
        return f"{int(x / 60)}"
def plot_tfc_condi_curve(df_stat, flag, plot_groups, width_mm, height_mm, colors, event_info, dpath_plot, title):
    """
    Plots a continuous freezing curve over time with SEM shading and event markers.
    
    plot_groups: List of group names to plot, in order (e.g., ['TFC_C_1mgKg', 'TFC_I_1mgKg']).
    event_info: Dictionary containing tone and shock timing.
    """
    set_pub_style() 
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')    
    # Sort the original column keys based on their actual numerical value
    if flag=='nTFC':
        time_cols = df_stat.columns[2:-1].values 
    if flag=='wTFC':
        time_cols = df_stat.columns[2:-3].values 
    # Create a strictly numerical array for Matplotlib's X-axis coordinates
    time_vals = np.array([float(t) for t in time_cols])
    
    metadata = {"Figure_Title": title, "Data_Summary": {}}
    # --- 2. PLOT EVENT SHADINGS (Tone & Shock) ---
    tone_starts = event_info.get('tone_start', [])
    tone_len = event_info.get('tone_len', 20)
    for start in tone_starts:
        ax.axvspan(start, start + tone_len, color='#d3d3d3', alpha=0.8, linewidth=0, zorder=1)
        
    shock_starts = event_info.get('shock_start', [])
    shock_len = event_info.get('shock_len', 2)
    for start in shock_starts:
        ax.axvspan(start, start + shock_len, facecolor='red', edgecolor='red', linewidth=0.1, alpha=1.0, zorder=1)
    
    # --- 3. PLOT GROUP CURVES ---
    for i, group in enumerate(plot_groups):       
        # Subset the dataframe using the exact original column names (time_cols)
        group_df = df_stat[df_stat['Group'] == group]
        data_matrix = group_df[time_cols].values.astype(float)
        
        n_mice = data_matrix.shape[0]       
        means = np.nanmean(data_matrix, axis=0)
        sems = stats.sem(data_matrix, axis=0, nan_policy='omit')        
        color = colors[i]
        
        # Plot using the numerical X-axis coordinates (time_vals)
        ax.plot(time_vals, means, color=color, linewidth=0.6, label=f"{group} (N={n_mice})", zorder=3)
        ax.fill_between(time_vals, means - sems, means + sems, color=color, alpha=0.2, linewidth=0, zorder=2)
        
        # Log metadata
        metadata["Data_Summary"][group] = {
            "N_mice": n_mice,
            "Final_Timepoint_Mean": float(means[-1]) if len(means) > 0 else 0,
            "Final_Timepoint_SEM": float(sems[-1]) if len(sems) > 0 else 0
        }
    # --- 4. AESTHETICS & FORMATTING ---
    ax.set_xlabel("Time (min)", labelpad=0.1)
    ax.set_ylabel("Freezing score (%)", labelpad=0.1)
    
    ax.set_ylim(0, 100)
    ax.yaxis.set_major_locator(ticker.MultipleLocator(20))  
    # 1. Force the X-axis to place a tick every 120 seconds (2 minutes)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(120))    
    # 2. Tell Matplotlib to divide the printed tick label by 60  
    ax.xaxis.set_major_formatter(FuncFormatter(format_to_minutes))
    # Set X limits based on raw seconds
    max_time = max(time_vals) if len(time_vals) > 0 else 690
    ax.set_xlim(0, max_time)   
    #ax.grid(axis='y', color='gray', linestyle='--', linewidth=0.5, alpha=0.3, zorder=0)
    
    # --- 5. SAVING OUTPUTS ---
    base_path = os.path.join(dpath_plot, title.replace(' ', '_'))  
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)
    plt.savefig(f"{base_path}.pdf")   #bbox_inches='tight', pad_inches=0.02
    plt.savefig(f"{base_path}.png", dpi=300) 
    plt.close()    
    save_metadata_json(metadata, dpath_plot, title)

def plot_combined_tfc_recall(df_stat, flag, plot_groups, width_mm, height_mm, colors_beh, event_info, dpath_plot, title):
    """
    Plots a continuous freezing curve (Left Panel) and epoch statistics (Right Panel) side-by-side.
    """
    set_pub_style() 
    # 1. Create a 1x2 grid. width_ratios=[4, 4] ensures both panels take exactly 50% of the space.
    fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(width_mm / 25.4, height_mm / 25.4),
                               gridspec_kw={'width_ratios': [4, 4]}, layout='constrained')   
    if flag == 'wTFC':
        time_cols = df_stat.columns[2:-7].values   
    else:
        time_cols = df_stat.columns[2:-1].values 
    time_vals = np.array([float(t) for t in time_cols])    
    metadata = {"Figure_Title": title, "Curve_Data": {}, "Statistics": {}}
    global_y_max = 100.0 # Will track the highest bracket in the stats plot 
    ax1.set_ylim(0, global_y_max) # For the correct distance between line and stars
    ax2.set_ylim(0, global_y_max)
    # ==========================================
    # PANEL 1: CONTINUOUS CURVE (ax1)
    # ==========================================
    # Event Shadings
    tone_starts = event_info.get('tone_start', [])
    tone_len = event_info.get('tone_len', 60) # Default to 60 for recall
    for start in tone_starts:
        ax1.axvspan(start, start + tone_len, color='#d3d3d3', alpha=0.8, linewidth=0, zorder=1)       
    # Group Curves
    for i, group in enumerate(plot_groups):        
        group_df = df_stat[df_stat['Group'] == group]
        data_matrix = group_df[time_cols].values.astype(float)
        
        n_mice = data_matrix.shape[0]        
        means = np.nanmean(data_matrix, axis=0)
        sems = stats.sem(data_matrix, axis=0, nan_policy='omit')        
        color = colors_beh[i]
        
        ax1.plot(time_vals, means, color=color, linewidth=0.6, label=f"{group} (N={n_mice})", zorder=3)
        ax1.fill_between(time_vals, means - sems, means + sems, color=color, alpha=0.2, linewidth=0, zorder=2)
        
        metadata["Curve_Data"][group] = {
            "N_mice": n_mice,
            "Final_Mean": float(means[-1]) if len(means) > 0 else 0,
            "Final_SEM": float(sems[-1]) if len(sems) > 0 else 0}
    # Ax1 Aesthetics
    ax1.set_xlabel("Time (min)", labelpad=0.1)
    ax1.set_ylabel("Freezing score (%)",  labelpad=0.1)    
    ax1.xaxis.set_major_locator(ticker.MultipleLocator(120))    
    ax1.xaxis.set_major_formatter(FuncFormatter(format_to_minutes))    
    max_time = max(time_vals) if len(time_vals) > 0 else 690
    ax1.set_xlim(0, max_time)    
    # ==========================================
    # PANEL 2: EPOCH STATISTICS (ax2)
    # ==========================================
    base_cols, tone_cols, ptone_cols = [], [], []
    base_start = event_info.get('base', 0)
    base_len = event_info.get('base_len', 120)      
    for t in time_cols:
        val = float(t)        
        if base_start <= val < base_start + base_len:
            base_cols.append(t)
        for start in event_info.get('tone_start', []):
            if start <= val < start + event_info.get('tone_len', 60):
                tone_cols.append(t)
        for start in event_info.get('p_tone_start', []):
            if start <= val < start + event_info.get('p_tone_len', 20):
                ptone_cols.append(t)                    
    epochs = ['Baseline', 'Tone', 'Post']
    epoch_col_map = [base_cols, tone_cols, ptone_cols]
    
    group_spacing = 1.2
    n_groups = len(plot_groups)    
    bar_width = 0.3
    offsets = np.linspace(-0.2, 0.2, n_groups) if n_groups == 2 else np.linspace(-0.3, 0.3, n_groups)
    
    for e_idx, epoch_name in enumerate(epochs):
        metadata["Statistics"][epoch_name] = {}
        base_x = e_idx * group_spacing        
        epoch_group_data = [] 
        epoch_group_x = []      
        
        for g_idx, group in enumerate(plot_groups):
            group_df = df_stat[df_stat['Group'] == group]
            cols = epoch_col_map[e_idx]            
            data_matrix = group_df[cols].values.astype(float)
            
            if data_matrix.size == 0: continue
                
            animal_means = np.nanmean(data_matrix, axis=1) 
            animal_means = animal_means[~np.isnan(animal_means)] 
            epoch_group_data.append(animal_means)
            
            n_mice = len(animal_means)
            if n_mice == 0: continue
            
            x_pos = base_x + offsets[g_idx]
            epoch_group_x.append(x_pos)
            color = colors_beh[g_idx]
            
            mean_val = np.mean(animal_means)
            sem_val = stats.sem(animal_means)            
            metadata["Statistics"][epoch_name][group] = {
                "N_mice": n_mice, "Mean": float(mean_val), "SEM": float(sem_val)}
            
            ax2.bar(x_pos, mean_val, yerr=sem_val, width=bar_width, 
                    facecolor='none', edgecolor=color, linewidth=0.75, capsize=0,
                    error_kw=dict(lw=0.75, ecolor='black'), zorder=2)          
            x_jitter = x_pos + np.random.uniform(-0.06, 0.06, size=n_mice)
            ax2.scatter(x_jitter, animal_means, color=color, edgecolor='none', 
                        s=2, zorder=3, alpha=1.0)
        
        # Apply Statistics
        if len(epoch_group_data) == 2 and len(epoch_group_data[0]) >= 3 and len(epoch_group_data[1]) >= 3:
            local_max = max(np.max(epoch_group_data[0]), np.max(epoch_group_data[1]))
            error_max = max(np.mean(epoch_group_data[0]) + stats.sem(epoch_group_data[0]), 
                            np.mean(epoch_group_data[1]) + stats.sem(epoch_group_data[1]))
            top_y = max(local_max, error_max)

            _, p_value = stats.mannwhitneyu(epoch_group_data[0], epoch_group_data[1], alternative='two-sided')
            metadata["Statistics"][epoch_name]['mannwhitneyu_p-values'] = float(p_value)
            
            bracket_top = add_stat_annotation_two_sided(
                ax2, epoch_group_data[0], epoch_group_data[1], 
                epoch_group_x[0], epoch_group_x[1], y_max=top_y, ttest=0, paired=0)
                
            if bracket_top is not None:
                global_y_max = max(global_y_max, bracket_top)

    # Ax2 Aesthetics
    ax2.set_xticks([i * group_spacing for i in range(len(epochs))])
    ax2.set_xticklabels(epochs, fontsize=7)
    #ax2.set_ylabel("Freezing score (%)",  labelpad=0.1)
    # ==========================================
    # FINAL SYNCHRONIZATION & EXPORT
    # ==========================================
    # Sync Y-axis so grids match perfectly across the figure
    ax1.set_ylim(0, global_y_max)
    ax2.set_ylim(0, global_y_max)
    
    ax1.yaxis.set_major_locator(ticker.MultipleLocator(20))
    ax2.yaxis.set_major_locator(ticker.MultipleLocator(20))

    base_path = os.path.join(dpath_plot, title.replace(' ', '_'))      
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)
    plt.savefig(f"{base_path}.pdf")   #bbox_inches='tight', pad_inches=0.02
    plt.savefig(f"{base_path}.png", dpi=300) 
    plt.close()          
    save_metadata_json(metadata, dpath_plot, title)
    
    
def plot_cfc_recall_stat(df_stat, plot_groups, width_mm, height_mm, colors_beh, data_col, dpath_plot, title):
    """
    Calculates the session-wide average freezing score for Contextual Fear Conditioning
    and plots it as a single grouped bar/scatter chart optimized for narrow widths.
    """
    set_pub_style() 
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')  
    # --- 2. PLOTTING SETUP ---
    n_groups = len(plot_groups)   
    
    # Dynamically center offsets based on number of groups 
    bar_width = 0.3
    offsets = np.linspace(-0.2, 0.2, n_groups) if n_groups == 2 else np.linspace(-0.3, 0.3, n_groups)
    ax.set_ylim(0, 100)
    
    metadata = {"Figure_Title": title, "Statistics": {"Context": {}}}       
    # --- 3. DATA EXTRACTION & PLOTTING ---
    # Nnly have one "epoch" (the entire context session), so base_x is simply 0
    base_x = 0    
    epoch_group_data = [] # Store data arrays for stats later
    epoch_group_x = []    # Store exact X coordinates for stats bracket       
    
    for g_idx, group in enumerate(plot_groups):
        group_df = df_stat[df_stat['Group'] == group]
        animal_means = group_df[data_col].values.astype(float)
        animal_means = animal_means[~np.isnan(animal_means)] # Drop true NaNs
        
        epoch_group_data.append(animal_means)
        
        n_mice = len(animal_means)
        if n_mice == 0: continue
        
        x_pos = base_x + offsets[g_idx]
        epoch_group_x.append(x_pos)
        color = colors_beh[g_idx]
        
        mean_val = np.mean(animal_means)
        sem_val = stats.sem(animal_means)                    
        # Record Metadata
        metadata["Statistics"]["Context"][group] = {
            "N_mice": n_mice, "Mean": float(mean_val), "SEM": float(sem_val)}
        
        # A. Draw Bar (Face='none', Edge=color)
        ax.bar(x_pos, mean_val, yerr=sem_val, width=bar_width, 
               facecolor='none', edgecolor=color, linewidth=0.75, capsize=0,
               error_kw=dict(lw=0.75, ecolor='black'), zorder=2)
        
        # B. Scatter Raw Data 
        x_jitter = x_pos + np.random.uniform(-0.06, 0.06, size=n_mice)
        ax.scatter(x_jitter, animal_means, color=color, edgecolor='none', 
                   s=2, zorder=3, alpha=1.0)
    
    # --- 4. APPLY STATISTICS ---
    # If have exactly 2 groups, test them against each other
    if len(epoch_group_data) == 2 and len(epoch_group_data[0]) >= 3 and len(epoch_group_data[1]) >= 3:
        local_max = max(np.max(epoch_group_data[0]), np.max(epoch_group_data[1]))
        error_max = max(np.mean(epoch_group_data[0]) + stats.sem(epoch_group_data[0]), 
                        np.mean(epoch_group_data[1]) + stats.sem(epoch_group_data[1]))
        top_y = max(local_max, error_max)

        _, p_value = stats.mannwhitneyu(epoch_group_data[0], epoch_group_data[1], alternative='two-sided')
        metadata["Statistics"]['mannwhitneyu_p-values'] = float(p_value)
        
        # Call the robust annotation function (using Mann-Whitney by default)
        add_stat_annotation_two_sided(
            ax, epoch_group_data[0], epoch_group_data[1], 
            epoch_group_x[0], epoch_group_x[1], y_max=top_y, ttest=0, paired=0)
            
    # --- 5. AESTHETICS & FORMATTING ---
    ax.set_xticks([base_x])
    ax.set_xticklabels(['Context'], fontsize=7)
    ax.set_ylabel("Freezing score (%)", labelpad=0.1)
    
    # Setup standard 0-100 scale
    
    ax.yaxis.set_major_locator(ticker.MultipleLocator(20))  
    
    # Force the X-axis limits to hug the bars tightly, preventing massive side margins
    ax.set_xlim(offsets[0] - 0.4, offsets[-1] + 0.4)     
    # --- 6. SAVING OUTPUTS ---
    base_path = os.path.join(dpath_plot, title.replace(' ', '_'))        
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)
    plt.savefig(f"{base_path}.pdf")   #bbox_inches='tight', pad_inches=0.02
    plt.savefig(f"{base_path}.png", dpi=300) 
    plt.close()        
    
    save_metadata_json(metadata, dpath_plot, title)


## 3.  Statistics of activity labeling of TFC

In [ ]:
dic_plots = {'Normal_I_1mgKg':  ['TFC_C_1mgKg', 'TFC_I_1mgKg'],
            'Normal_E_0.2mgKg': ['TFC_C_0.2mgKg', 'TFC_E_0.2mgKg']}
height_mm =25
idx_plot = 0
for key, plot_groups in dic_plots.items():   
    idx_plot +=1
    if 'I' in key:
        colors = colors_beh_i
    elif 'E' in key:
        colors = colors_beh_e

    df_stat = pd.read_csv(os.path.join(dir_stat, "02_4.TFC_activity_labeling.csv")) 
    width_mm  = 20 
    # CA1 activity labeling
    plot_activity_stat(df_stat, plot_groups, width_mm, height_mm, colors, 'CA1_engram', dpath_plot, f'03_{str(idx_plot)}_1_activity_{key}_CA1')
    # EC3 activity labeling
    width_mm  = 22
    plot_activity_stat(df_stat, plot_groups, width_mm, height_mm, colors, 'MEC_engram', dpath_plot, f'03_{str(idx_plot)}_2_activity_{key}_EC3')

print('All finished')

In [ ]:
def plot_activity_stat(df_stat, plot_groups, width_mm, height_mm, colors_beh, data_col, dpath_plot, title):
    """
    Calculates the session-wide average freezing score for Contextual Fear Conditioning
    and plots it as a single grouped bar/scatter chart optimized for narrow widths.
    """
    set_pub_style() 
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')  
    # --- 2. PLOTTING SETUP ---
    n_groups = len(plot_groups)        
    # Dynamically center offsets based on number of groups 
    bar_width = 0.3
    offsets = np.linspace(-0.2, 0.2, n_groups) if n_groups == 2 else np.linspace(-0.3, 0.3, n_groups)
    
    metadata = {"Figure_Title": title, "Statistics": {"Context": {}}}    
    # --- 3. DATA EXTRACTION & PLOTTING ---
    # Only have one "epoch" (the entire context session), so base_x is simply 0
    base_x = 0    
    epoch_group_data = [] # Store data arrays for stats later
    epoch_group_x = []    # Store exact X coordinates for stats bracket        
    global_y_max = 0   
    for g_idx, group in enumerate(plot_groups):
        group_df = df_stat[df_stat['Group'] == group]
        animal_means = group_df[data_col].values.astype(float)
        animal_means = animal_means[~np.isnan(animal_means)] # Drop true NaNs
        
        epoch_group_data.append(animal_means)
        
        n_mice = len(animal_means)
        if n_mice == 0: continue
        
        x_pos = base_x + offsets[g_idx]
        epoch_group_x.append(x_pos)
        color = colors_beh[g_idx]
        
        mean_val = np.mean(animal_means)
        sem_val = stats.sem(animal_means)                    
        # Record Metadata
        metadata["Statistics"]["Context"][group] = {
            "N_mice": n_mice, "Mean": float(mean_val), "SEM": float(sem_val)}
        
        # A. Draw Bar (Face='none', Edge=color)
        ax.bar(x_pos, mean_val, yerr=sem_val, width=bar_width, 
               facecolor='none', edgecolor=color, linewidth=1.0, capsize=0,
               error_kw=dict(lw=0.75, ecolor='black'), zorder=2)
        
        # B. Scatter Raw Data 
        x_jitter = x_pos + np.random.uniform(-0.06, 0.06, size=n_mice)
        ax.scatter(x_jitter, animal_means, color=color, edgecolor='none', 
                   s=2, zorder=3, alpha=1.0)
    
    # --- 4. APPLY STATISTICS ---
    # If have exactly 2 groups, test them against each other
    if len(epoch_group_data) == 2 and len(epoch_group_data[0]) >= 3 and len(epoch_group_data[1]) >= 3:
        local_max = max(np.max(epoch_group_data[0]), np.max(epoch_group_data[1]))
        error_max = max(np.mean(epoch_group_data[0]) + stats.sem(epoch_group_data[0]), 
                        np.mean(epoch_group_data[1]) + stats.sem(epoch_group_data[1]))
        top_y = max(local_max, error_max)
        
        # Call the robust annotation function
        bracket_top = add_stat_annotation_two_sided(ax, epoch_group_data[0], epoch_group_data[1], 
            epoch_group_x[0], epoch_group_x[1], y_max=top_y, ttest=0, paired=0)
        
        if bracket_top is not None:
            global_y_max = max(global_y_max, bracket_top)
            
        # Log 2-tailed t-test p-value to metadata
        stat, p_val = stats.mannwhitneyu(epoch_group_data[0], epoch_group_data[1])
        metadata["Statistics"]["Context"]["mannwhitneyu_P_Value"] = float(p_val)
        metadata["Statistics"]["Context"]["mannwhitneyu_stat"] = float(stat)        
    # --- 5. AESTHETICS & FORMATTING ---
    ax.set_xticks([base_x])
    ax.set_xticklabels(['Context'], fontsize=7)
    if 'CA1' in data_col:
        ax.set_ylabel("Memory trace\n(cells/mm)", labelpad=0.1)
    if 'MEC' in data_col:
        ax.set_ylabel("Memory trace\n(cells/mm^2)", labelpad=0.1)
    
    # Setup standard 0-100 scale
    ax.set_ylim(0, global_y_max)
    #ax.yaxis.set_major_locator(ticker.MultipleLocator(20))  
    ax.set_xlim(offsets[0] - 0.4, offsets[-1] + 0.4)    
    # --- 6. SAVING OUTPUTS ---
    base_path = os.path.join(dpath_plot, title.replace(' ', '_'))        
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)
    plt.savefig(f"{base_path}.pdf")   #bbox_inches='tight', pad_inches=0.02
    plt.savefig(f"{base_path}.png", dpi=300)  
    plt.close()        
    
    save_metadata_json(metadata, dpath_plot, title)